# TFM — JSRT + Partial MaxSAT: ejecución integral en Kaggle

Ejecute **Run All**. El cuaderno clona el repositorio, instala las dependencias, obtiene los dos datasets JSRT públicos, prepara la anotación de nódulo, ejecuta las cuatro oclusiones, genera resultados y los publica en GitHub si existe el secreto `GITHUB_TOKEN`.

Antes de ejecutar, active **Internet** en Kaggle. Para publicar, cree el secreto `GITHUB_TOKEN` con permiso GitHub **Contents: Read and write** para `ttromas-arch/tfm-maxsat-xai`. Sin secreto, el cuaderno completa el benchmark y deja el ZIP en `/kaggle/working`, pero no puede autenticar un push a GitHub.

> **Rigor del GT:** `masks/` son máscaras de pulmón. El metadata complementario aporta centro y tamaño clínico de los 154 nódulos, no contornos manuales por píxel. Por tanto, el GT de nódulo se reconstruye como círculo y se registra como `circle_from_coordinate_diameter`.

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_URL = 'https://github.com/ttromas-arch/tfm-maxsat-xai.git'
REPOSITORY_SLUG = 'ttromas-arch/tfm-maxsat-xai'
LUNG_DATASET_REF = 'abduzzami/jsrt-247-image-lung-segmentation-mask-dataset'
METADATA_DATASET_REF = 'raddar/nodules-in-chest-xrays-jsrt'
WORKING_ROOT = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
REPOSITORY_DIR = WORKING_ROOT / 'tfm_maxsat_xai'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
PREPARED_DIR = WORKING_ROOT / 'jsrt_prepared'
RESULTS_DIR = WORKING_ROOT / 'tfm_results'
PUBLISHED_DIR = WORKING_ROOT / 'kaggle_artifacts' / RUN_ID

# Full benchmark defaults. Lower MAX_IMAGES only for a development run.
MAX_IMAGES = None
NUM_SUPERPIXELS = 30
MAX_ORACLE_CALLS = 100
DIVERSE_SOLUTIONS = 1
ORACLE_BATCH_SIZE = 16
PER_SAMPLE_TIMEOUT_SEC = 300
SAVE_SUCCESSFUL_EXAMPLES = 5
WEIGHT_MODE = 'area'
# Relative 0.80 means: reduce each image's original Nodule score by at least 20%.
# This is robust to the absolute calibration of a multi-label CXR classifier.
THRESHOLD = 0.80
THRESHOLD_MODE = 'relative'
SEED = 42

def run(command: list[str], cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print('+', ' '.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

# A second Run All in the same Kaggle session must not reuse partial CSV markers.
for path in (PREPARED_DIR, RESULTS_DIR, PUBLISHED_DIR):
    shutil.rmtree(path, ignore_errors=True)

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(REPOSITORY_DIR)])
os.chdir(REPOSITORY_DIR)
print(f'Run ID: {RUN_ID}')
print(f'Repository commit: {subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()}')

In [ ]:
# Keep Kaggle's preinstalled torch/torchvision build, including its CUDA support.
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '-r', 'requirements-kaggle.txt'])
run([sys.executable, '-c', 'import cv2, pysat, skimage, torch, torchxrayvision; print({"torch": torch.__version__, "cuda": torch.cuda.is_available(), "xrv": torchxrayvision.__version__})'])

In [ ]:
from src.data.kaggle_jsrt import (
    LUNG_MASK_DATASET_REF,
    NODULE_METADATA_DATASET_REF,
    discover_lung_mask_layout,
    find_jsrt_metadata_csv,
)

def mounted_root_with_lung_layout() -> Path | None:
    for candidate in sorted(path for path in INPUT_ROOT.iterdir() if path.is_dir()):
        try:
            discover_lung_mask_layout(candidate)
            return candidate
        except (FileNotFoundError, ValueError):
            continue
    return None

def mounted_root_with_metadata() -> Path | None:
    for candidate in sorted(path for path in INPUT_ROOT.iterdir() if path.is_dir()):
        try:
            find_jsrt_metadata_csv(candidate)
            return candidate
        except FileNotFoundError:
            continue
    return None

lung_root = mounted_root_with_lung_layout()
metadata_root = mounted_root_with_metadata()
if lung_root is None or metadata_root is None:
    import kagglehub
    if lung_root is None:
        print(f'Downloading {LUNG_MASK_DATASET_REF} through kagglehub')
        lung_root = Path(kagglehub.dataset_download(LUNG_MASK_DATASET_REF))
    if metadata_root is None:
        print(f'Downloading {NODULE_METADATA_DATASET_REF} through kagglehub')
        metadata_root = Path(kagglehub.dataset_download(NODULE_METADATA_DATASET_REF))

print('lung dataset root:', lung_root)
print('metadata dataset root:', metadata_root)
images, masks = discover_lung_mask_layout(lung_root)
metadata_csv = find_jsrt_metadata_csv(metadata_root)
print('cxr:', images)
print('lung masks:', masks)
print('nodule metadata:', metadata_csv)

In [ ]:
# Build an auditable canonical annotation table in the writable Kaggle area.
run([
    sys.executable, 'scripts/prepare_kaggle_jsrt.py',
    '--lung-dataset-root', str(lung_root),
    '--metadata-dataset-root', str(metadata_root),
    '--output-dir', str(PREPARED_DIR),
])

import json
manifest_path = PREPARED_DIR / 'kaggle_jsrt_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['matched_pair_count'] == 247, manifest
assert manifest['annotated_nodule_count'] == 154, manifest
print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
command = [
    sys.executable, 'scripts/run_experiments.py',
    '--jsrt-root', str(lung_root),
    '--image-dir', manifest['image_dir'],
    '--lung-mask-dir', manifest['lung_mask_dir'],
    '--nodule-csv', manifest['nodule_csv'],
    '--provenance-manifest', str(manifest_path),
    '--ground-truth-representation', manifest['nodule_gt_representation'],
    '--output-dir', str(RESULTS_DIR),
    '--num-superpixels', str(NUM_SUPERPIXELS),
    '--weight-mode', WEIGHT_MODE,
    '--threshold', str(THRESHOLD),
    '--threshold-mode', THRESHOLD_MODE,
    '--max-oracle-calls', str(MAX_ORACLE_CALLS),
    '--diverse-solutions', str(DIVERSE_SOLUTIONS),
    '--oracle-batch-size', str(ORACLE_BATCH_SIZE),
    '--per-sample-timeout-sec', str(PER_SAMPLE_TIMEOUT_SEC),
    '--save-successful-examples', str(SAVE_SUCCESSFUL_EXAMPLES),
    '--seed', str(SEED),
    '--overwrite',
]
if MAX_IMAGES is not None:
    command.extend(['--max-images', str(MAX_IMAGES)])
run(command)

In [ ]:
import pandas as pd
from IPython.display import display

results_csv = RESULTS_DIR / 'results.csv'
results = pd.read_csv(results_csv)
assert not results.empty
assert results['ground_truth_representation'].eq('circle_from_coordinate_diameter').all()
display(results.groupby('occlusion_method')[['validity', 'iou_nodule', 'dice_nodule', 'num_occluded_superpixels', 'total_time_sec']].agg(['mean', 'median', 'count']))
print('Rows:', len(results), '| Valid explanations:', int(results['validity'].sum()))
print('Results:', RESULTS_DIR)

# Show the first available patient panel inline.
from IPython.display import Image as DisplayImage
panels = sorted((RESULTS_DIR / 'figures').glob('*_panel.png'))
if panels:
    display(DisplayImage(filename=str(panels[0])))

In [ ]:
# Publish a compact, reproducible artifact set. Full artifacts remain in RESULTS_DIR and the ZIP.
PUBLISHED_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['results.csv', 'summary.md', 'summary.tex', 'config.json', 'ground_truth_provenance.json', 'completed_methods.jsonl', 'failures.jsonl']:
    source = RESULTS_DIR / filename
    if source.is_file():
        shutil.copy2(source, PUBLISHED_DIR / filename)
for filename in ['boxplots.png', 'violins.png']:
    source = RESULTS_DIR / filename
    if source.is_file():
        shutil.copy2(source, PUBLISHED_DIR / filename)
shutil.copy2(manifest_path, PUBLISHED_DIR / 'kaggle_jsrt_manifest.json')
run_manifest = {
    'run_id': RUN_ID,
    'repository': REPOSITORY_SLUG,
    'repository_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'lung_dataset_ref': LUNG_DATASET_REF,
    'metadata_dataset_ref': METADATA_DATASET_REF,
    'results_rows': int(len(results)),
    'validity_sum': int(results['validity'].sum()),
    'ground_truth_representation': manifest['nodule_gt_representation'],
}
(PUBLISHED_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2) + '\n', encoding='utf-8')
archive = shutil.make_archive(str(WORKING_ROOT / 'tfm_maxsat_artifacts'), 'zip', root_dir=PUBLISHED_DIR.parent, base_dir=PUBLISHED_DIR.name)
print('Artifact directory:', PUBLISHED_DIR)
print('Artifact ZIP:', archive)

In [ ]:
# Push results safely to a new branch when the optional Kaggle secret exists.
def github_token_from_kaggle() -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception as error:
        print(f'GitHub publication skipped: GITHUB_TOKEN is unavailable ({error}).')
        return None

token = github_token_from_kaggle()
if token:
    branch = f'kaggle-results/{RUN_ID}'
    destination = REPOSITORY_DIR / 'kaggle_artifacts' / RUN_ID
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(PUBLISHED_DIR, destination, dirs_exist_ok=False)
    run(['git', 'checkout', '-B', branch], cwd=REPOSITORY_DIR)
    run(['git', 'add', '-f', str(destination.relative_to(REPOSITORY_DIR))], cwd=REPOSITORY_DIR)
    run(['git', 'config', 'user.name', 'kaggle-tfm-bot'], cwd=REPOSITORY_DIR)
    run(['git', 'config', 'user.email', 'kaggle-tfm-bot@users.noreply.github.com'], cwd=REPOSITORY_DIR)
    run(['git', 'commit', '-m', f'Kaggle experiment artifacts {RUN_ID}'], cwd=REPOSITORY_DIR)
    askpass = WORKING_ROOT / '.github_askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf \'%s\\n\' x-access-token ;; *) printf \'%s\\n\' "$GITHUB_TOKEN" ;; esac\n', encoding='utf-8')
    askpass.chmod(0o700)
    push_env = os.environ.copy()
    push_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': token})
    try:
        run(['git', 'push', '--set-upstream', 'origin', branch], cwd=REPOSITORY_DIR, env=push_env)
        print(f'Published: https://github.com/{REPOSITORY_SLUG}/tree/{branch}')
    finally:
        askpass.unlink(missing_ok=True)
else:
    print('Benchmark completed. Download /kaggle/working/tfm_maxsat_artifacts.zip or configure GITHUB_TOKEN and Run All again to publish.')